# 05 — Retrieval Evaluation

This notebook evaluates the visual product retrieval system developed in the
previous notebooks.

The evaluation uses the existing ResNet-50 embeddings and FAISS index to
measure retrieval quality across the product catalog.

### Evaluation objectives

- Define a reproducible relevance criterion using product metadata
- Generate evaluation queries from the catalog
- Retrieve Top-K visually similar products
- Calculate Recall@K
- Calculate Precision@K
- Calculate Mean Average Precision (mAP)
- Inspect retrieval examples
- Analyze the limitations of the evaluation methodology

### Evaluation pipeline

```text
Product Catalog
      ↓
Evaluation Query
      ↓
FAISS Similarity Search
      ↓
Top-K Retrieved Products
      ↓
Metadata-Based Relevance
      ↓
Recall@K / Precision@K / mAP

## 1- Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from tqdm.auto import tqdm

print("Imports successful")

Imports successful


d:\Projects\VisualProductSearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2- Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"

METADATA_PATH = PROCESSED_DIR / "product_metadata.csv"
EMBEDDINGS_PATH = EMBEDDINGS_DIR / "product_embeddings.npy"
EMBEDDING_IDS_PATH = EMBEDDINGS_DIR / "embedding_product_ids.npy"
FAISS_INDEX_PATH = EMBEDDINGS_DIR / "product_embeddings_faiss.index"

print("Project root:", PROJECT_ROOT)
print("Metadata:", METADATA_PATH)
print("Embeddings:", EMBEDDINGS_PATH)
print("Embedding IDs:", EMBEDDING_IDS_PATH)
print("FAISS index:", FAISS_INDEX_PATH)

Project root: d:\Projects\VisualProductSearch
Metadata: d:\Projects\VisualProductSearch\data\processed\product_metadata.csv
Embeddings: d:\Projects\VisualProductSearch\embeddings\product_embeddings.npy
Embedding IDs: d:\Projects\VisualProductSearch\embeddings\embedding_product_ids.npy
FAISS index: d:\Projects\VisualProductSearch\embeddings\product_embeddings_faiss.index


## 3- Load Metadata

In [3]:
metadata_df = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata_df.shape)
display(metadata_df.head())

Metadata shape: (44441, 11)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,d:\Projects\VisualProductSearch\data\fashion-p...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,d:\Projects\VisualProductSearch\data\fashion-p...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,d:\Projects\VisualProductSearch\data\fashion-p...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,d:\Projects\VisualProductSearch\data\fashion-p...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,d:\Projects\VisualProductSearch\data\fashion-p...


## 4- Load Embeddings and Product IDs

In [4]:
embeddings = np.load(EMBEDDINGS_PATH)
embedding_product_ids = np.load(EMBEDDING_IDS_PATH)

print("Embeddings shape:", embeddings.shape)
print("Product ID mapping shape:", embedding_product_ids.shape)
print("Embedding dtype:", embeddings.dtype)
print("ID dtype:", embedding_product_ids.dtype)

Embeddings shape: (44441, 2048)
Product ID mapping shape: (44441,)
Embedding dtype: float32
ID dtype: int64


## 5- Load FAISS Index

In [5]:
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

print("FAISS index loaded")
print("Number of vectors:", faiss_index.ntotal)
print("Vector dimension:", faiss_index.d)

FAISS index loaded
Number of vectors: 44441
Vector dimension: 2048


## 6- Validate Alignment

In [6]:
assert embeddings.shape[0] == len(embedding_product_ids)
assert embeddings.shape[0] == faiss_index.ntotal
assert embeddings.shape[1] == faiss_index.d
assert len(metadata_df) == len(embedding_product_ids)

print("✓ Embedding count matches product-ID mapping")
print("✓ Embedding count matches FAISS index")
print("✓ Embedding dimension matches FAISS index")
print("✓ Metadata count matches embedding mapping")

✓ Embedding count matches product-ID mapping
✓ Embedding count matches FAISS index
✓ Embedding dimension matches FAISS index
✓ Metadata count matches embedding mapping


## 7- Create Product Lookup

In [7]:
metadata_lookup = metadata_df.set_index("id")

print("Metadata lookup created")

Metadata lookup created


## 8- Check Article Types

In [8]:
article_type_counts = metadata_df["articleType"].value_counts()

print("Number of unique article types:", article_type_counts.shape[0])

display(article_type_counts.head(20))

Number of unique article types: 142


articleType
Tshirts                  7069
Shirts                   3215
Casual Shoes             2846
Watches                  2542
Sports Shoes             2036
Kurtas                   1844
Tops                     1762
Handbags                 1759
Heels                    1323
Sunglasses               1073
Wallets                   936
Flip Flops                916
Sandals                   897
Briefs                    849
Belts                     813
Backpacks                 724
Socks                     686
Formal Shoes              637
Perfume and Body Mist     614
Jeans                     608
Name: count, dtype: int64

## 9- Define Relevance

For the initial evaluation, a retrieved product is considered relevant when
its `articleType` matches the query product's `articleType`.

For example:

```text
Query:
articleType = T-Shirts

Retrieved:
T-Shirt       → Relevant
T-Shirt       → Relevant
Jeans         → Not relevant
Shirts        → Not relevant

In [9]:
K_VALUES = [5, 10]

NUM_EVAL_QUERIES = 1000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

print("K values:", K_VALUES)
print("Number of evaluation queries:", NUM_EVAL_QUERIES)
print("Random seed:", RANDOM_SEED)

K values: [5, 10]
Number of evaluation queries: 1000
Random seed: 42


## 10- Select Evaluation Queries

In [10]:
MIN_PRODUCTS_PER_TYPE = 11

eligible_types = (
    metadata_df["articleType"]
    .value_counts()
)

eligible_types = eligible_types[
    eligible_types >= MIN_PRODUCTS_PER_TYPE
].index

eligible_mask = metadata_df["articleType"].isin(eligible_types)

eligible_indices = np.flatnonzero(eligible_mask.to_numpy())

print("Eligible article types:", len(eligible_types))
print("Eligible products:", len(eligible_indices))

Eligible article types: 107
Eligible products: 44287


## 11- Sample Queries

In [11]:
num_queries = min(NUM_EVAL_QUERIES, len(eligible_indices))

query_indices = rng.choice(
    eligible_indices,
    size=num_queries,
    replace=False
)

print("Evaluation queries selected:", len(query_indices))

Evaluation queries selected: 1000


## 12- Retrieval Function

The FAISS index stores L2-normalized embeddings and uses inner-product
similarity.

For normalized vectors:

```text
inner product = cosine similarity

In [12]:
def retrieve_similar_products(query_index, k):
    """
    Retrieve Top-K products for a query while excluding the query product.

    The query embedding is L2-normalized so that FAISS inner-product
    scores correspond to cosine similarity.
    """

    query_vector = embeddings[query_index:query_index + 1].copy()

    faiss.normalize_L2(query_vector)

    search_k = k + 1

    scores, indices = faiss_index.search(
        query_vector,
        search_k
    )

    scores = scores[0]
    indices = indices[0]

    mask = indices != query_index

    filtered_indices = indices[mask][:k]
    filtered_scores = scores[mask][:k]

    return filtered_indices, filtered_scores

## 13- Test Retrieval

In [13]:
test_query_index = query_indices[0]

retrieved_indices, retrieved_scores = retrieve_similar_products(
    test_query_index,
    k=10
)

query_product_id = embedding_product_ids[test_query_index]

print("Query product ID:", query_product_id)
print("Retrieved indices:", retrieved_indices)
print("Retrieved scores:", retrieved_scores)

Query product ID: 57166
Retrieved indices: [26340 22449 27270 21920 27678 41896 44073 26117 13854 44282]
Retrieved scores: [0.9200197  0.919512   0.907767   0.90605974 0.90448487 0.90346944
 0.9020921  0.90175176 0.9014675  0.89916724]


## 14- Display Test Retrieval

In [14]:
query_row = metadata_df.iloc[test_query_index]

print("Query:")
print("Product ID:", query_row["id"])
print("Article Type:", query_row["articleType"])
print("Product Name:", query_row["productDisplayName"])

results_df = metadata_df.iloc[retrieved_indices].copy()
results_df["similarity_score"] = retrieved_scores

display(
    results_df[
        ["id", "articleType", "baseColour", "productDisplayName", "similarity_score"]
    ]
)

Query:
Product ID: 57166
Article Type: Tops
Product Name: Elle Women Lavender Top


,id,articleType,baseColour,productDisplayName,similarity_score
26340,36739,Tshirts,White,Spykar Women Cult White T-shirt,0.920020
22449,36738,Tshirts,White,Spykar Women Cult White T-shirt,0.919512
27270,11525,Tops,White,United Colors of Benetton Women White Top,0.907767
21920,21156,Shirts,Grey,s.Oliver Women White Striped Shirt,0.906060
27678,41404,Tops,Pink,Wrangler Women Stud Pink Top,0.904485
41896,40198,Tops,White,Mumbai Slang Women White Printed Top,0.903469
44073,22322,Tshirts,Blue,Wildcraft Women Solid Blue Tshirt,0.902092
26117,42673,Kurtas,White,Alma Women White Kurta,0.901752
13854,37916,Tops,White,Mineral Women White Top,0.901468
44282,50705,Tops,Cream,Latin Quarters Women Cream Printed Top,0.899167


## 15- Create Relevance Function

In [15]:
def get_relevance_labels(query_index, retrieved_indices):
    """
    Return binary relevance labels based on matching articleType.

    1 = relevant
    0 = not relevant
    """

    query_article_type = metadata_df.iloc[query_index]["articleType"]

    retrieved_article_types = metadata_df.iloc[
        retrieved_indices
    ]["articleType"].to_numpy()

    relevance = (
        retrieved_article_types == query_article_type
    ).astype(np.int32)

    return relevance

## 16- Test Relevance

In [16]:
relevance = get_relevance_labels(
    test_query_index,
    retrieved_indices
)

print("Query article type:", query_row["articleType"])
print("Relevance labels:", relevance)

Query article type: Tops
Relevance labels: [0 0 1 0 1 1 0 0 1 1]


## 17- Recall@K

Recall@K measures how many relevant products were retrieved within the
Top-K results compared with the total number of relevant products available
in the catalog, excluding the query itself.

```text
Recall@K =
relevant retrieved in Top-K
---------------------------
total relevant products

In [17]:
def recall_at_k(query_index, retrieved_indices, k):
    query_article_type = metadata_df.iloc[query_index]["articleType"]

    total_relevant = (
        (metadata_df["articleType"] == query_article_type).sum() - 1
    )

    if total_relevant <= 0:
        return np.nan

    retrieved_relevant = get_relevance_labels(
        query_index,
        retrieved_indices[:k]
    ).sum()

    return retrieved_relevant / total_relevant

## 18- Precision@K

Precision@K measures the proportion of retrieved products in the Top-K that
are relevant.

```text
Precision@K =
relevant products in Top-K
--------------------------
K

In [18]:
def precision_at_k(query_index, retrieved_indices, k):
    relevance = get_relevance_labels(
        query_index,
        retrieved_indices[:k]
    )

    return relevance.sum() / k

## 19- Average Precision

Average Precision considers the ranking position of relevant results.

A relevant result appearing earlier in the ranking contributes more strongly
to the score than a relevant result appearing later.

In [19]:
def average_precision(query_index, retrieved_indices, k):
    relevance = get_relevance_labels(
        query_index,
        retrieved_indices[:k]
    )

    num_relevant = 0
    precision_sum = 0.0

    for rank, is_relevant in enumerate(relevance, start=1):

        if is_relevant:
            num_relevant += 1
            precision_sum += num_relevant / rank

    total_relevant = (
        (metadata_df["articleType"] == metadata_df.iloc[query_index]["articleType"]).sum()
        - 1
    )

    denominator = min(total_relevant, k)

    if denominator == 0:
        return np.nan

    return precision_sum / denominator

## 20- Test Metrics on One Query 

In [20]:
for k in K_VALUES:
    recall = recall_at_k(
        test_query_index,
        retrieved_indices,
        k
    )

    precision = precision_at_k(
        test_query_index,
        retrieved_indices,
        k
    )

    ap = average_precision(
        test_query_index,
        retrieved_indices,
        k
    )

    print(f"K={k}")
    print(f"  Recall@{k}:    {recall:.4f}")
    print(f"  Precision@{k}: {precision:.4f}")
    print(f"  AP@{k}:        {ap:.4f}")

K=5
  Recall@5:    0.0011
  Precision@5: 0.4000
  AP@5:        0.1467
K=10
  Recall@10:    0.0028
  Precision@10: 0.5000
  AP@10:        0.2178


## 21- Full Catalog Evaluation

The following cell evaluates the selected query set.

For every query:

1. Retrieve Top-10 products using FAISS.
2. Remove the query product.
3. Determine relevance using `articleType`.
4. Calculate Recall@5 and Recall@10.
5. Calculate Precision@5 and Precision@10.
6. Calculate AP@5 and AP@10.

In [21]:
evaluation_records = []

for query_index in tqdm(query_indices, desc="Evaluating queries"):

    retrieved_indices, retrieved_scores = retrieve_similar_products(
        query_index,
        k=max(K_VALUES)
    )

    record = {
        "query_index": query_index,
        "query_product_id": embedding_product_ids[query_index],
        "article_type": metadata_df.iloc[query_index]["articleType"],
    }

    for k in K_VALUES:

        record[f"recall@{k}"] = recall_at_k(
            query_index,
            retrieved_indices,
            k
        )

        record[f"precision@{k}"] = precision_at_k(
            query_index,
            retrieved_indices,
            k
        )

        record[f"ap@{k}"] = average_precision(
            query_index,
            retrieved_indices,
            k
        )

    evaluation_records.append(record)

evaluation_df = pd.DataFrame(evaluation_records)

print("Evaluation completed")
print("Evaluation shape:", evaluation_df.shape)

display(evaluation_df.head())

Evaluating queries: 100%|██████████| 1000/1000 [00:36<00:00, 27.62it/s]

Evaluation completed
Evaluation shape: (1000, 9)


,query_index,query_product_id,article_type,recall@5,precision@5,ap@5,recall@10,precision@10,ap@10
0,22271,57166,Tops,0.001136,0.4,0.146667,0.002839,0.5,0.217778
1,41776,33455,Handbags,0.002844,1.0,1.000000,0.004551,0.8,0.732778
2,15672,14701,Trousers,0.009452,1.0,1.000000,0.018904,1.0,1.000000
3,22865,50975,Bra,0.010504,1.0,1.000000,0.021008,1.0,1.000000
4,27192,18007,Backpacks,0.006916,1.0,1.000000,0.013831,1.0,1.000000


## 22- Calculate Overall Metrics

In [22]:
metrics = {
    "Recall@5": evaluation_df["recall@5"].mean(),
    "Recall@10": evaluation_df["recall@10"].mean(),
    "Precision@5": evaluation_df["precision@5"].mean(),
    "Precision@10": evaluation_df["precision@10"].mean(),
    "mAP@5": evaluation_df["ap@5"].mean(),
    "mAP@10": evaluation_df["ap@10"].mean(),
}

metrics_df = pd.DataFrame(
    metrics.items(),
    columns=["Metric", "Score"]
)

display(metrics_df)

,Metric,Score
0,Recall@5,0.008484
1,Recall@10,0.014570
2,Precision@5,0.786400
3,Precision@10,0.755300
4,mAP@5,0.746647
5,mAP@10,0.700968


### Print Results

In [23]:
print("======================================")
print("      VISUAL RETRIEVAL EVALUATION")
print("======================================")

for metric, score in metrics.items():
    print(f"{metric:<15}: {score:.4f}")

print("--------------------------------------")
print("Evaluation queries:", len(evaluation_df))
print("Relevance proxy: articleType")

      VISUAL RETRIEVAL EVALUATION
Recall@5       : 0.0085
Recall@10      : 0.0146
Precision@5    : 0.7864
Precision@10   : 0.7553
mAP@5          : 0.7466
mAP@10         : 0.7010
--------------------------------------
Evaluation queries: 1000
Relevance proxy: articleType


## 23- Category-Level Performance

In [30]:
category_metrics = (
    evaluation_df
    .groupby("article_type")
    .agg(
        queries=("query_product_id", "count"),
        recall_at_5=("recall@5", "mean"),
        recall_at_10=("recall@10", "mean"),
        precision_at_5=("precision@5", "mean"),
        precision_at_10=("precision@10", "mean"),
        map_at_5=("ap@5", "mean"),
        map_at_10=("ap@10", "mean"),
    )
)

category_metrics = (
    category_metrics[
        category_metrics["queries"] >= 10
    ]
    .sort_values("map_at_10")
)

display(category_metrics.head(20))

,queries,recall_at_5,recall_at_10,precision_at_5,precision_at_10,map_at_5,map_at_10
article_type,,,,,,,
Dresses,14,0.003085,0.006017,0.285714,0.278571,0.175476,0.149736
Flats,11,0.003461,0.006376,0.345455,0.318182,0.274545,0.200740
Tops,29,0.001136,0.002252,0.400000,0.396552,0.298851,0.262146
Nightdress,10,0.013830,0.024468,0.520000,0.460000,0.400000,0.327960
Sweaters,13,0.009755,0.017280,0.538462,0.476923,0.458462,0.373919
Heels,31,0.002464,0.004392,0.651613,0.580645,0.587419,0.494667
Perfume and Body Mist,15,0.005764,0.009897,0.706667,0.606667,0.643111,0.527643
Kurtas,41,0.001959,0.003706,0.721951,0.682927,0.669024,0.597750
Sandals,18,0.004278,0.008371,0.766667,0.750000,0.682037,0.636204


## 24- Retrieval Failure Analysis

Quantitative metrics provide an overall measurement of retrieval quality,
but they do not explain why the model succeeds or fails.

The next cells identify queries with low Precision@10 so that their retrieved
products can be inspected manually.

In [25]:
poor_queries = (
    evaluation_df
    .sort_values("precision@10")
    .head(10)
)

display(poor_queries)

,query_index,query_product_id,article_type,recall@5,precision@5,ap@5,recall@10,precision@10,ap@10
954,43094,11932,Flats,0.0,0.0,0.0,0.0,0.0,0.0
432,10939,47670,Scarves,0.0,0.0,0.0,0.0,0.0,0.0
424,16392,32808,Heels,0.0,0.0,0.0,0.0,0.0,0.0
946,20717,17664,Casual Shoes,0.0,0.0,0.0,0.0,0.0,0.0
541,36972,57030,Tops,0.0,0.0,0.0,0.0,0.0,0.0
64,40913,12770,Tops,0.0,0.0,0.0,0.0,0.0,0.0
59,30042,58098,Pendant,0.0,0.0,0.0,0.0,0.0,0.0
68,30434,7612,Messenger Bag,0.0,0.0,0.0,0.0,0.0,0.0
567,36340,21882,Tunics,0.0,0.0,0.0,0.0,0.0,0.0
94,30880,33962,Tunics,0.0,0.0,0.0,0.0,0.0,0.0


## 25- Inpect a Poor Query

In [26]:
poor_query_index = int(
    poor_queries.iloc[0]["query_index"]
)

poor_query_retrieved_indices, poor_query_scores = retrieve_similar_products(
    poor_query_index,
    k=10
)

poor_query_row = metadata_df.iloc[poor_query_index]

print("Query Product")
print("-----------------------------")
print("Product ID:", poor_query_row["id"])
print("Article Type:", poor_query_row["articleType"])
print("Product Name:", poor_query_row["productDisplayName"])

poor_results = metadata_df.iloc[poor_query_retrieved_indices].copy()
poor_results["similarity_score"] = poor_query_scores
poor_results["relevant"] = get_relevance_labels(
    poor_query_index,
    poor_query_retrieved_indices
)

display(
    poor_results[
        [
            "id",
            "articleType",
            "baseColour",
            "productDisplayName",
            "similarity_score",
            "relevant"
        ]
    ]
)

Query Product
-----------------------------
Product ID: 11932
Article Type: Flats
Product Name: Skechers Women Tone Black Sandals


,id,articleType,baseColour,productDisplayName,similarity_score,relevant
21678,16185,Formal Shoes,Black,Enroute Men Leather Black Formal Shoes,0.930456,0
16727,55629,Casual Shoes,Brown,Numero Uno Men Brown Shoes,0.928815,0
20615,55628,Casual Shoes,Black,Numero Uno Men Black Shoes,0.925814,0
15309,57495,Casual Shoes,Brown,Lee Cooper Men Brown Shoes,0.924324,0
12819,12855,Formal Shoes,Black,Lee Cooper Men Formal Black Shoes,0.923178,0
13950,37975,Casual Shoes,Black,Clarks Men Black Leather Casual Shoes,0.921441,0
43211,55625,Formal Shoes,Brown,Numero Uno Men Brown Formal Shoes,0.920928,0
44129,57947,Casual Shoes,Black,Clarks Men Black Shoes,0.920711,0
9614,20714,Casual Shoes,Coffee Brown,Red Tape Men Coffee Brown Leather Loafers,0.919796,0
27710,22156,Formal Shoes,Black,Lee Cooper Men Black Formal Shoes,0.919126,0


## 26- Save Evaluation Results

In [27]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

EVALUATION_RESULTS_PATH = RESULTS_DIR / "retrieval_evaluation.csv"
CATEGORY_RESULTS_PATH = RESULTS_DIR / "category_level_evaluation.csv"

evaluation_df.to_csv(
    EVALUATION_RESULTS_PATH,
    index=False
)

category_metrics.to_csv(
    CATEGORY_RESULTS_PATH
)

print("Saved:")
print(EVALUATION_RESULTS_PATH)
print(CATEGORY_RESULTS_PATH)

Saved:
d:\Projects\VisualProductSearch\results\retrieval_evaluation.csv
d:\Projects\VisualProductSearch\results\category_level_evaluation.csv


## 27- Save Summary Metrics

In [28]:
METRICS_PATH = RESULTS_DIR / "retrieval_metrics.csv"

metrics_df.to_csv(
    METRICS_PATH,
    index=False
)

print("Metrics saved to:", METRICS_PATH)

Metrics saved to: d:\Projects\VisualProductSearch\results\retrieval_metrics.csv


## 28- Final Validation

In [29]:
assert not evaluation_df.empty

for column in [
    "recall@5",
    "recall@10",
    "precision@5",
    "precision@10",
    "ap@5",
    "ap@10"
]:
    assert evaluation_df[column].between(0, 1).all()

print("✓ Evaluation results are non-empty")
print("✓ All metric values are within [0, 1]")
print("✓ Evaluation completed successfully")

✓ Evaluation results are non-empty
✓ All metric values are within [0, 1]
✓ Evaluation completed successfully


# Evaluation Summary

The visual product retrieval system was evaluated using the ResNet-50
embeddings and FAISS similarity-search index.

### Evaluation configuration

- Evaluation queries: 1,000 or fewer when fewer eligible products are available
- Retrieval method: FAISS `IndexFlatIP`
- Similarity: cosine similarity through L2-normalized embeddings
- K values: 5 and 10
- Relevance criterion: matching `articleType`
- Query product excluded from retrieved results

### Metrics

The evaluation reports:

- Recall@5
- Recall@10
- Precision@5
- Precision@10
- mAP@5
- mAP@10

### Important limitation

The dataset does not provide human-annotated visual similarity labels.
Therefore, `articleType` is used as a proxy for relevance.

Consequently, these metrics should be interpreted as measurements of
same-product-type retrieval rather than direct measurements of human-rated
visual similarity.

The results will be used as the baseline for later model comparisons and
error analysis.